# TP Nº N 5 Título del práctico

**Asignatura:** IMT-342 Robótica  
**Gestión:** II-2026  
**Docente:** Bernardo Quiroga Turdera  

**Integrantes:**  
- Roberth Williams Ruiz Condori  
- Mario Alberto Nina Gallo  

**Fuente oficial:** [guía o documento correspondiente]

## Ejercicio 1

### Enunciado

[Consigna del ejercicio]

### Desarrollo

## Ejercicio 3 y 4 — Implementación Simbólica, Numérica y Validación (CDIO)

### Enunciado

Desarrolle el script `dh_builder.py` con las funciones `dh_matrix_symbolic`, `dh_matrix_numeric`, y `fk_chain`.

Para una junta con $\theta_1 = 45^\circ$, $d_1 = 0.35\,m$, $a_1 = 0.15\,m$, $\alpha_1 = -90^\circ$: extraiga $R$ y demuestre por código que

$$
\|R^T R-I\|_F < 10^{-15},
$$

$$
|\det(R)-1.0| < 10^{-15},
$$

y

$$
p =
\begin{bmatrix}
a_1\cos\theta_1\\
a_1\sin\theta_1\\
d_1
\end{bmatrix}.
$$

## Ejercicio 3 — Implementación Simbólica y Numérica

In [1]:
import numpy as np
import sympy as sp

from dh_builder import (
    dh_matrix_symbolic,
    dh_matrix_numeric,
    fk_chain
)

### 3.1 Matriz D-H simbólica

In [2]:
theta, d, a, alpha = sp.symbols(
    'theta d a alpha',
    real=True
)

T_symbolic = dh_matrix_symbolic(
    theta, d, a, alpha
)

sp.pprint(T_symbolic)

⎡cos(θ)  -sin(θ)⋅cos(α)  sin(α)⋅sin(θ)   a⋅cos(θ)⎤
⎢                                                ⎥
⎢sin(θ)  cos(α)⋅cos(θ)   -sin(α)⋅cos(θ)  a⋅sin(θ)⎥
⎢                                                ⎥
⎢  0         sin(α)          cos(α)         d    ⎥
⎢                                                ⎥
⎣  0           0               0            1    ⎦


### 3.2 Matriz D-H numérica

Se evalúa la matriz D-H para los parámetros indicados en el enunciado.
Los ángulos se convierten de grados a radianes para trabajar con NumPy.

In [3]:
theta1 = np.deg2rad(45)
d1 = 0.35
a1 = 0.15
alpha1 = np.deg2rad(-90)

T_numeric = dh_matrix_numeric(
    theta1,
    d1,
    a1,
    alpha1
)

print("Matriz D-H numérica:")
print(T_numeric)

Matriz D-H numérica:
[[ 7.07106781e-01 -4.32978028e-17 -7.07106781e-01  1.06066017e-01]
 [ 7.07106781e-01  4.32978028e-17  7.07106781e-01  1.06066017e-01]
 [ 0.00000000e+00 -1.00000000e+00  6.12323400e-17  3.50000000e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


### 3.3 Cinemática directa mediante `fk_chain`

Se construye una cadena D-H con los parámetros de la junta y se obtiene la transformación homogénea resultante mediante la multiplicación encadenada.

In [4]:
dh_params = [
    (theta1, d1, a1, alpha1)
]

T_chain = fk_chain(dh_params)

print("Transformación obtenida mediante fk_chain:")
print(T_chain)

Transformación obtenida mediante fk_chain:
[[ 7.07106781e-01 -4.32978028e-17 -7.07106781e-01  1.06066017e-01]
 [ 7.07106781e-01  4.32978028e-17  7.07106781e-01  1.06066017e-01]
 [ 0.00000000e+00 -1.00000000e+00  6.12323400e-17  3.50000000e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [5]:
chain_error = np.linalg.norm(
    T_chain - T_numeric
)

print("Error entre ambas transformaciones:")
print(chain_error)

Error entre ambas transformaciones:
0.0


## Ejercicio 4 — Validación Numérica

### 4.1 Extracción de la matriz de rotación y del vector de posición

A partir de la transformación homogénea obtenida, se extraen la matriz de rotación $R$ y el vector de posición $p$.

In [6]:
R = T_numeric[:3, :3]
p = T_numeric[:3, 3]

print("Matriz de rotación R:")
print(R)

print("\nVector de posición p:")
print(p)

Matriz de rotación R:
[[ 7.07106781e-01 -4.32978028e-17 -7.07106781e-01]
 [ 7.07106781e-01  4.32978028e-17  7.07106781e-01]
 [ 0.00000000e+00 -1.00000000e+00  6.12323400e-17]]

Vector de posición p:
[0.10606602 0.10606602 0.35      ]


### 4.2 Validación de ortogonalidad

Una matriz de rotación debe satisfacer:

$$
R^T R = I
$$

Se calcula el error mediante la norma de Frobenius y se verifica que sea menor que $10^{-15}$.

In [7]:
I = np.eye(3)

orthogonality_error = np.linalg.norm(
    R.T @ R - I,
    ord='fro'
)

print("Error de ortogonalidad:")
print(orthogonality_error)

assert orthogonality_error < 1e-15

Error de ortogonalidad:
1.434936932798653e-17


### 4.3 Validación del determinante

Se verifica que la matriz de rotación pertenezca a $SO(3)$ mediante:

$$
\det(R)=1
$$

y se comprueba que el error respecto a $1$ sea menor que $10^{-15}$.

In [8]:
det_R = np.linalg.det(R)
det_error = abs(det_R - 1.0)

print("det(R):")
print(det_R)

print("\nError del determinante:")
print(det_error)

assert det_error < 1e-15

det(R):
1.0

Error del determinante:
0.0


### 4.4 Validación del vector de posición

Se compara el vector de posición obtenido de la matriz homogénea con el valor esperado:

$$
p =
\begin{bmatrix}
a_1\cos\theta_1\\
a_1\sin\theta_1\\
d_1
\end{bmatrix}.
$$

In [9]:
p_expected = np.array([
    a1 * np.cos(theta1),
    a1 * np.sin(theta1),
    d1
])

position_error = np.linalg.norm(
    p - p_expected
)

print("Posición obtenida:")
print(p)

print("\nPosición esperada:")
print(p_expected)

print("\nError de posición:")
print(position_error)

assert position_error < 1e-15

Posición obtenida:
[0.10606602 0.10606602 0.35      ]

Posición esperada:
[0.10606602 0.10606602 0.35      ]

Error de posición:
0.0


### 4.5 Resultado de la validación

Las pruebas numéricas cumplen las condiciones establecidas en la guía:

- Error de ortogonalidad:
  $\|R^T R-I\|_F = 1.4349\times10^{-17} < 10^{-15}$.
- Determinante:
  $\det(R)=1.0$.
- Error del determinante:
  $|\det(R)-1.0|=0 < 10^{-15}$.
- El vector de posición obtenido coincide con:
  $p=[a_1\cos\theta_1,\ a_1\sin\theta_1,\ d_1]^T$.

Por tanto, la transformación homogénea D-H implementada supera las validaciones numéricas solicitadas.

## Conclusión

Se implementó el módulo `dh_builder.py` utilizando SymPy y NumPy, incorporando las funciones `dh_matrix_symbolic`, `dh_matrix_numeric` y `fk_chain` para construir y encadenar transformaciones homogéneas mediante la convención D-H estándar.

La implementación fue validada para $\theta_1=45^\circ$, $d_1=0.35\,m$, $a_1=0.15\,m$ y $\alpha_1=-90^\circ$. Los resultados verificaron la ortogonalidad de la matriz de rotación, obteniendo $\|R^T R-I\|_F=1.4349\times10^{-17}<10^{-15}$, así como $\det(R)=1.0$. Además, el vector de posición obtenido coincidió con la expresión teórica indicada en la guía.

Por tanto, la implementación cumple las condiciones numéricas establecidas y permite utilizar las funciones desarrolladas para construir transformaciones D-H de forma simbólica, numérica y encadenada.